# VarChat — İkinci yargıç: `qwen2.5:14b` (Google Colab)

`colab_olcum.ipynb` üç modelin cevaplarını üretti ve `qwen2.5:7b` yargıçla puanladı. Ama 7B yargıç hem küçük hem de
`qwen-taban` ile aynı model ailesinden: kendi yazım tarzını kayırıyor olabilir. Bu defter **aynı cevapları** (Drive'daki
`varchat-olcum/sonuclar`) daha güçlü bir yargıçla, `qwen2.5:14b` ile **yeniden puanlar**. Cevaplar yeniden üretilmez.

Sonunda iki tablo çıkar: 14B yargıca göre karşılaştırma ve iki yargıcın aynı cümlelerde ne kadar anlaştığı (Cohen kappa).
7B yargıcın sonuçları silinmez; 14B'ninkiler `__yargic-qwen2.5-14b` ekli ayrı dosyalara yazılır.

**Kullanım:** *Runtime → Change runtime type → **L4*** (T4 de olur, ~2 kat yavaş) → *Runtime → **Run all***. Drive iznini onayla.
Süre: L4'te tahminen 1-1,5 saat. Oturum koparsa yeniden *Run all*: biten modeller tekrarlanmaz.

## 0) GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1) Projeyi getir + paketler

In [ ]:
import os
if not os.path.exists('/content/varchat'):
    !git clone -q https://github.com/atu-ce/varchat.git /content/varchat
else:
    !cd /content/varchat && git fetch -q origin && git checkout -q origin/main -- . ':(exclude)degerlendirme/sonuclar'
%cd /content/varchat
!git log --oneline -1 origin/main
!pip install -q "ollama==0.6.2" "rapidfuzz==3.14.5"

## 2) Drive'a bağlan (cevaplar `colab_olcum` defterinden)

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
SONUC = '/content/drive/MyDrive/varchat-olcum/sonuclar'
assert os.path.exists(SONUC), "Drive'da varchat-olcum/sonuclar yok: önce colab_olcum.ipynb çalışmış olmalı"
yerel = '/content/varchat/degerlendirme/sonuclar'
if not os.path.islink(yerel):
    for ad in os.listdir(yerel):
        kaynak = os.path.join(yerel, ad)
        if os.path.isfile(kaynak) and not os.path.exists(os.path.join(SONUC, ad)):
            shutil.copy2(kaynak, SONUC)
    shutil.rmtree(yerel)
    os.symlink(SONUC, yerel)
uretimler = sorted(a for a in os.listdir(SONUC) if a.startswith("uretim_"))
print("bulunan üretim dosyaları:", uretimler)
assert any("varchat" in a for a in uretimler) and any("qwen-taban" in a for a in uretimler), "qwen-taban / varchat cevapları eksik"

## 3) Ollama + ikinci yargıç modeli (`qwen2.5:14b`, ~9 GB)

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests, json, datetime

def ollama_hazir():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=2).json()
    except Exception:
        return None

if not ollama_hazir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_hazir():
            break
assert ollama_hazir(), "Ollama başlamadı; /content/ollama.log dosyasına bak"
!ollama pull qwen2.5:14b
!ollama run qwen2.5:14b "Merhaba. Tek kelimeyle cevap ver."
ps_cikti = !ollama ps
print("\n".join(ps_cikti))
if not any("GPU" in s for s in ps_cikti[1:]):
    print("UYARI: model GPU'da değil; Runtime type GPU mu?")
gpu = !nvidia-smi --query-gpu=name --format=csv,noheader
liste = !ollama list
kod_surumu = !git -C /content/varchat log --oneline -1
json.dump({"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "gpu": list(gpu), "ollama": ollama_hazir(),
           "modeller": list(liste), "kod": list(kod_surumu)},
          open(f"{SONUC}/ortam_yargic14b_{datetime.date.today().isoformat()}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## 4) Yeniden puanla: `qwen-taban` ve `varchat` (asıl karşılaştırma)

Aynı yargıç istemi ve ayarları (sıcaklık 0); yalnızca yargıç modeli değişir.

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen-taban --yargic qwen2.5:14b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model varchat --yargic qwen2.5:14b

## 5) (İsteğe bağlı) Ham `qwen2.5:7b`'yi de puanla

Asıl karşılaştırma için gerekmez; zaman varsa tamlık için. Atlamak istersen bu hücreyi çalıştırma.

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen2.5:7b --yargic qwen2.5:14b

## 6) Sonuçlar: 14B yargıca göre karşılaştırma + iki yargıcın uyumu

- **Sadakat farkı (qwen-taban → varchat)** ve %95 güven aralığı: 14B yargıca göre eğitim sadakati artırdı mı, azalttı mı?
- **Yargıç uyumu:** iki yargıç aynı cümlelerde ne sıklıkla aynı kararı verdi. Kappa 0.6 üstü iyi, 0.4 altı zayıf uyum demektir.

In [ ]:
!python degerlendirme/b08_karsilastir.py --yargic qwen2.5:14b --uyum qwen2.5:14b

## 7) Sonuçları bilgisayarına al

Drive'daki **`varchat-olcum/sonuclar`** klasöründen adında **`__yargic-qwen2.5-14b`** geçen dosyaları ve
`karsilastirma_istatistik_*`, `ortam_yargic14b_*` dosyalarını indirip projede `degerlendirme/colab_son/` içine koy.